# SARIMAX 04B — Rolling-Origin Efficient Training

This notebook runs the revised SARIMAX implementation **without replacing the original model**.

The revised version:

- reuses the specification selected by the original tuning run;
- fits one model per FSA and fold instead of one model per horizon;
- updates the state without refitting at each forecast origin;
- reports convergence, progress, elapsed time, and ETA;
- writes outputs to separate `sarimax_rolling` folders.

Always run the smoke test first.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# Define the root directory of the project and config file
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "sarimax_rolling.yaml"
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/sarimax_rolling.yaml')

In [3]:
# Import functions to run SARIMAX model
from src.ontario_peak_risk.models.forecasting.sarimax.run_model_rolling import (
    run_sarimax_rolling,
)


## Step 1 — Smoke test

This fits only the configured representative fold/FSA and forecasts a small number of origins.

Use the output to verify:

- whether the optimizer converges;
- fit duration;
- forecasting speed;
- whether the revised pipeline works before committing to the complete run.

In [ ]:
# Run test
smoke_results = run_sarimax_rolling(
    CONFIG_PATH,
    mode="smoke",
    resume=False,
)


SARIMAX ROLLING-ORIGIN REVISION
Mode: smoke
Specification: {'order': [2, 0, 1], 'seasonal_order': [1, 0, 0, 24], 'trend': 'c'}
Strategy: one fit per FSA/fold + rolling state updates
Tasks to process: 1
Final 2025 holdout evaluated: NO
Smoke test does not create the official final metrics files.

[Task 1/1] fold_2023 / L4T
  Fit data: 17,520 hourly observations (2021-01-01 00:00:00 -> 2022-12-31 23:00:00)
  Fit completed | time=2m 31s | converged=True | iterations=64 | warnings=0
  Forecast progress | 1/48 origins (2.1%) | elapsed=0s | task ETA=4s
  Forecast progress | 48/48 origins (100.0%) | elapsed=0s | task ETA=0s
  Task checkpoint saved: fold_2023__L4T.parquet
[Task 1/1] completed | task time=2m 33s | estimated remaining run time=0s

SMOKE TEST COMPLETED
Elapsed: 2m 33s
Review convergence and runtime before running mode='full'.


In [5]:
display(smoke_results["fit_diagnostics"])
display(smoke_results["global_metrics"])


,fold,fsa,training_observations,validation_origins,forecast_rows,specification,requested_trend,implemented_trend,explicit_intercept_in_exog,converged,fit_seconds,iterations,convergence_warning_count,aic,bic,task_seconds,resumed_from_checkpoint
0,fold_2023,L4T,17520,48,1152,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,151.945812,64,0,240016.302638,240117.30761,152.472533,False


,model,mae,rmse,mape,wape,bias,n_observations
0,sarimax_rolling,427.818224,574.083419,3.986423,4.0147,13.633556,1152


## Step 2 — Full development evaluation

Run this cell **only after reviewing the smoke-test output**.

The complete run processes all development folds and FSAs. Completed FSA/fold tasks are checkpointed. If the kernel is interrupted later, rerunning with `resume=True` will skip completed tasks.

In [6]:
# Run the model
rolling_results = run_sarimax_rolling(
    CONFIG_PATH,
    mode="full",
    resume=True,
)


SARIMAX ROLLING-ORIGIN REVISION
Mode: full
Specification: {'order': [2, 0, 1], 'seasonal_order': [1, 0, 0, 24], 'trend': 'c'}
Strategy: one fit per FSA/fold + rolling state updates
Tasks to process: 12
Final 2025 holdout evaluated: NO

[Task 1/12] fold_2023 / L4T
  Fit data: 17,520 hourly observations (2021-01-01 00:00:00 -> 2022-12-31 23:00:00)
  Fit completed | time=1m 57s | converged=True | iterations=64 | warnings=0
  Forecast progress | 1/8,736 origins (0.0%) | elapsed=0s | task ETA=2m 46s
  Forecast progress | 250/8,736 origins (2.9%) | elapsed=2s | task ETA=1m 08s
  Forecast progress | 500/8,736 origins (5.7%) | elapsed=4s | task ETA=1m 07s
  Forecast progress | 750/8,736 origins (8.6%) | elapsed=6s | task ETA=1m 05s
  Forecast progress | 1,000/8,736 origins (11.4%) | elapsed=8s | task ETA=1m 03s
  Forecast progress | 1,250/8,736 origins (14.3%) | elapsed=10s | task ETA=1m 01s
  Forecast progress | 1,500/8,736 origins (17.2%) | elapsed=12s | task ETA=59s
  Forecast progress | 1,

In [7]:
# Display results
display(rolling_results["fit_diagnostics"])



,fold,fsa,training_observations,validation_origins,forecast_rows,specification,requested_trend,implemented_trend,explicit_intercept_in_exog,converged,fit_seconds,iterations,convergence_warning_count,aic,bic,task_seconds,resumed_from_checkpoint
0,fold_2023,L4T,17520,8736,209664,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,117.146457,64,0,240016.302638,240117.307610,188.088755,False
1,fold_2023,M5R,17520,8736,209664,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,68.579175,32,0,225559.488045,225660.493017,137.153361,False
2,fold_2023,M5S,17520,8736,209664,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,45.400665,23,0,203643.145559,203744.150532,114.381006,False
3,fold_2023,M6G,17520,8736,209664,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,54.847965,32,0,235779.163958,235880.168930,125.110818,False
4,fold_2023,M9R,17520,8736,209664,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,99.721255,67,0,228145.458441,228246.463414,168.057463,False
5,fold_2023,M9W,17520,8736,209664,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,93.066968,58,0,239950.229903,240051.234875,171.550297,False
6,fold_2024,L4T,26280,8760,210240,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,125.744056,56,0,358601.813878,358708.096336,218.067452,False
7,fold_2024,M5R,26280,8760,210240,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,137.941613,54,0,338670.975588,338777.258045,216.184814,False
8,fold_2024,M5S,26280,8760,210240,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,100.964099,26,0,305306.769936,305413.052393,245.104464,False
9,fold_2024,M6G,26280,8760,210240,"{'order': [2, 0, 1], 'seasonal_order': [1, 0, ...",c,n,True,True,158.087482,28,0,353715.326939,353821.609397,261.567204,False


In [8]:
# Displya Global Metric
display(rolling_results["global_metrics"])


,model,mae,rmse,mape,wape,bias,n_observations
0,sarimax_rolling,583.635455,929.486636,6.089956,6.41415,-164.621729,2519424


In [9]:
# Display Flod metrics
display(rolling_results["fold_metrics"])

,fold,mae,rmse,mape,wape,bias,n_observations
0,fold_2023,598.344296,953.983324,6.190622,6.630155,-196.007804,1257984
1,fold_2024,568.966912,904.396505,5.989566,6.202236,-133.321644,1261440
